# 16 — 4-Way Regional: RoBERTa-Large × 3 Seeds

Same task and split as notebook 11 — only the backbone changes from RoBERTa-base (~125M params) to RoBERTa-large (~355M params). Backbone capacity is a plausible bottleneck on the regional task; large variants typically add +2–3 macro-F1 points on similar text classification tasks.

**Hyperparameters changed for the larger model:**
- `learning_rate`: 1e-5 (down from 2e-5; large models prefer lower LRs)
- `per_device_train_batch_size`: 8 (down from 16 to fit memory)
- `gradient_accumulation_steps`: 4 (up from 2 — keeps effective batch 32 same as NB 11)
- `warmup_ratio`: 0.1 (up from 0.06)
- everything else identical to NB 11

Reference (NB 11, RoBERTa-base): F1 0.817 ± 0.007, accuracy 0.834 ± 0.005.


In [1]:
import os
import re
import json
import random
import numpy as np
import pandas as pd
import torch

from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    balanced_accuracy_score, f1_score,
)
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from transformers import (
    AutoModelForSequenceClassification, AutoTokenizer, DataCollatorWithPadding,
    EarlyStoppingCallback, Trainer, TrainingArguments,
)
from transformers.utils.notebook import NotebookProgressCallback

RANDOM_STATE = 42
SEEDS = [42, 123, 2024]
TEXT_COLUMN = 'text_full_concat_scrubbed_plus'
MAX_LENGTH = 256

TRAIN_BATCH_SIZE = 8
EVAL_BATCH_SIZE = 16
GRAD_ACCUM_STEPS = 4
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.1
USE_FP16 = True

ROBERTA_LARGE_CKPT = 'roberta-large'
ROBERTA_LARGE_LR = 1e-5

OUTPUT_DIR_ROOT = 'artifacts/origin_region_4way_roberta_large'
os.makedirs(OUTPUT_DIR_ROOT, exist_ok=True)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)

def set_seed(seed: int):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

COUNTRY_TO_REGION = {
    # --- East Africa ---
    'Ethiopia': 'East Africa',
    'Kenya': 'East Africa',
    'Rwanda': 'East Africa',
    'Burundi': 'East Africa',
    'Tanzania': 'East Africa',
    'Uganda': 'East Africa',
    'DR Congo': 'East Africa',
    'Zambia': 'East Africa',
    'Zimbabwe': 'East Africa',
    'Cameroon': 'East Africa',
    'Malawi': 'East Africa',
    'South Africa': 'East Africa',
    'Yemen': 'East Africa',
    # --- Central America ---
    'Guatemala': 'Central America',
    'Costa Rica': 'Central America',
    'Panama': 'Central America',
    'El Salvador': 'Central America',
    'Honduras': 'Central America',
    'Nicaragua': 'Central America',
    'Mexico': 'Central America',
    'Jamaica': 'Central America',
    'Puerto Rico': 'Central America',
    'Haiti': 'Central America',
    'Dominican Republic': 'Central America',
    # --- South America ---
    'Colombia': 'South America',
    'Peru': 'South America',
    'Brazil': 'South America',
    'Ecuador': 'South America',
    'Bolivia': 'South America',
    'Venezuela': 'South America',
    # --- Asia-Pacific ---
    'Indonesia': 'Asia-Pacific',
    'Taiwan': 'Asia-Pacific',
    'Thailand': 'Asia-Pacific',
    'Papua New Guinea': 'Asia-Pacific',
    'Philippines': 'Asia-Pacific',
    'India': 'Asia-Pacific',
    'Vietnam': 'Asia-Pacific',
    'China': 'Asia-Pacific',
    'Timor-Leste': 'Asia-Pacific',
    'Malaysia': 'Asia-Pacific',
    'Laos': 'Asia-Pacific',
    'Nepal': 'Asia-Pacific',
    'Myanmar': 'Asia-Pacific',
    'Australia': 'Asia-Pacific',
    'United Kingdom': 'Asia-Pacific',
    'United States': 'Asia-Pacific',
}


Device: cuda


## Scrubbing vocabulary (identical to NB 11)


In [2]:
# Scrubbing vocabulary: countries + region aliases + cultivars + producer context
COUNTRY_TERMS = {
    "Ethiopia": ["ethiopia", "ethiopian"],
    "Colombia": ["colombia", "colombian", "columbian"],
    "Panama": ["panama", "panamanian"],
    "Kenya": ["kenya", "kenyan"],
    "Indonesia": ["indonesia", "indonesian"],
    "Guatemala": ["guatemala", "guatemalan"],
    "Costa Rica": ["costa rica", "costa rican", "costarican"],
    "El Salvador": ["el salvador", "salvadoran", "salvadorean", "salvadorian"],
    "Rwanda": ["rwanda", "rwandan"],
    "Brazil": ["brazil", "brazilian"],
    "Honduras": ["honduras", "honduran"],
    "Peru": ["peru", "peruvian"],
    "Taiwan": ["taiwan", "taiwanese"],
    "Papua New Guinea": ["papua new guinea", "papua new guinean"],
    "Nicaragua": ["nicaragua", "nicaraguan"],
    "Burundi": ["burundi", "burundian"],
    "Thailand": ["thailand", "thai"],
    "India": ["india", "indian"],
    "Mexico": ["mexico", "mexican"],
    "Tanzania": ["tanzania", "tanzanian"],
    "Yemen": ["yemen", "yemeni"],
    "Ecuador": ["ecuador", "ecuadorian"],
    "Bolivia": ["bolivia", "bolivian"],
    "Jamaica": ["jamaica", "jamaican"],
    "Uganda": ["uganda", "ugandan"],
    "Dominican Republic": ["dominican republic", "dominican"],
    "Zambia": ["zambia", "zambian"],
    "China": ["china", "chinese"],
    "Vietnam": ["vietnam", "vietnamese"],
    "Philippines": ["philippines", "philippine", "filipino"],
    "Malaysia": ["malaysia", "malaysian"],
    "Laos": ["laos", "laotian"],
    "Zimbabwe": ["zimbabwe", "zimbabwean"],
    "Haiti": ["haiti", "haitian"],
    "Puerto Rico": ["puerto rico", "puerto rican", "puertorican"],
    "Nepal": ["nepal", "nepalese", "nepali"],
    "Myanmar": ["myanmar", "burmese"],
    "Cameroon": ["cameroon", "cameroonian"],
    "Australia": ["australia", "australian"],
    "South Africa": ["south africa", "south african"],
    "Malawi": ["malawi", "malawian"],
    "Venezuela": ["venezuela", "venezuelan", "merida state", "mocoties valley"],
    "Timor-Leste": ["east timor", "timor leste", "timorese"],
    "DR Congo": ["democratic republic of the congo", "dr congo", "congo", "congolese"],
    "United Kingdom": ["united kingdom", "british", "pitcairn island", "saint helena", "st helena", "sandy bay valley"],
    "United States": [
        "usa", "united states", "american",
        "hawaii", "hawai'i", "hawaiian", "hawaii island",
        "big island", "kona", "puna district", "holualoa", "oahu", "maui", "kauai",
        "ka u", "ka'u", "kau",
    ],
}

REGION_ALIASES = {
    "Ethiopia": ["yirgacheffe", "sidamo", "sidama", "guji", "gedeb", "gedeo", "kochere", "hambela",
                 "shakiso", "jimma", "limu", "oromia", "harrar", "kaffa", "bench maji", "bench-maji",
                 "arbegona", "bensa"],
    "Colombia": ["huila", "cauca", "narino", "tolima", "quindio", "caldas", "risaralda", "antioquia",
                 "cundinamarca", "santander", "pitalito", "acevedo", "planadas", "gaitania", "piendamo",
                 "caicedonia", "san agustin", "armenia"],
    "Panama": ["boquete", "chiriqui", "volcan", "jaramillo", "alto quiel", "paso ancho",
               "piedra candela", "canas verdes", "silla del pando", "renacimiento"],
    "Kenya": ["nyeri", "kirinyaga", "kiambu", "embu", "muranga", "murang'a", "thika", "ruiru",
              "mathira", "meru", "nakuru", "gichugu", "karatina"],
    "Indonesia": ["sumatra", "aceh", "gayo", "lintong", "mandheling", "sidikalang", "toraja",
                  "sulawesi", "java", "bali", "kintamani", "flores", "kerinci"],
    "Guatemala": ["huehuetenango", "antigua", "acatenango", "fraijanes", "coban", "atitlan",
                  "chimaltenango", "quiche", "solola", "palencia", "sacatepequez", "san marcos",
                  "hoja blanca", "cuilco"],
    "Costa Rica": ["tarrazu", "central valley", "west valley", "tres rios", "naranjo", "dota",
                   "poas", "alajuela", "brunca", "turrialba", "coto brus", "chirripo"],
    "El Salvador": ["ahuachapan", "chalatenango", "apaneca", "ilamatepec", "santa ana",
                    "el boqueron", "quetzaltepec", "juayua", "ataco"],
    "Rwanda": ["gakenke", "nyamasheke", "karongi", "huye", "nyamagabe", "rulindo", "gikongoro",
               "lake kivu"],
    "Brazil": ["cerrado", "mogiana", "minas gerais", "mantiqueira", "sul de minas",
               "chapada diamantina", "carmo de minas"],
    "Honduras": ["marcala", "copan", "ocotepeque", "comayagua", "intibuca", "santa barbara",
                 "el paraiso", "capucas"],
    "Peru": ["cajamarca", "jaen", "san ignacio", "chanchamayo", "cusco", "villa rica", "oxapampa",
             "junin", "puno"],
    "Taiwan": ["alishan", "yunlin", "chiayi", "chia yi", "nantou", "taichung", "pingtung"],
    "Papua New Guinea": ["wahgi valley", "western highlands", "eastern highlands", "jiwaka",
                         "kainantu", "okapa"],
    "Nicaragua": ["jinotega", "matagalpa", "nueva segovia", "madriz", "ocotal", "dipilto"],
    "Burundi": ["kayanza", "ngozi", "muramvya", "muyinga", "bururi"],
    "Thailand": ["chiang rai", "doi chang", "doi pangkhon", "chiang mai", "nan province"],
    "India": ["coorg", "chikmagalur", "karnataka", "bababudangiri", "nilgiris"],
    "Mexico": ["chiapas", "oaxaca", "veracruz", "coatepec", "pluma hidalgo"],
    "Tanzania": ["mbeya", "ruvuma", "ngorongoro", "arusha", "kilimanjaro", "mbozi"],
    "Yemen": ["haraaz", "haraz", "sanaa", "bani matar", "hayma"],
    "Ecuador": ["loja", "pichincha", "imbabura", "zamora", "chimborazo", "saraguro"],
    "Bolivia": ["caranavi", "yungas", "la paz"],
    "Jamaica": ["blue mountain", "blue mountains"],
    "Uganda": ["rwenzori", "bugisu", "sipi falls"],
    "Vietnam": ["lam dong", "quang tri", "dalat", "cau dat"],
    "Philippines": ["benguet", "bukidnon", "davao"],
    "China": ["yunnan", "baoshan", "puer", "pu'er", "lincong"],
    "Puerto Rico": ["utuado", "yauco", "adjuntas"],
    "Venezuela": ["mocoties valley", "merida state"],
    "Malawi": ["malawi"],
}

REGION_ONLY_TERMS = [
    "central america", "south america", "latin america",
    "east africa", "central africa", "west africa", "africa",
    "asia", "the americas", "americas",
    "central and south america", "south and central america",
    "east and central africa", "various africa growing regions",
]

CULTIVAR_TERMS = [
    "sl28", "sl-28", "sl 28", "sl34", "sl-34", "sl 34",
    "ruiru 11", "ruiru-11", "batian", "k7",
    "gesha", "geisha",
    "pacamara", "pache", "villa sarchi",
    "bourbon", "pink bourbon", "yellow bourbon", "red bourbon",
    "caturra", "catuai", "catuai vermelho", "catuai amarelo",
    "typica", "mundo novo",
    "maragogipe", "maragogype", "maracaturra",
    "castillo", "colombia variety", "variedad colombia", "tabi",
    "heirloom", "ethiopian heirloom",
    "tim tim", "s795", "ateng",
    "catimor", "sarchimor", "icatu", "obata",
]

PRODUCER_CONTEXT_TERMS = [
    "finca", "hacienda", "beneficio",
    "cup of excellence",
    "washing station", "wet mill",
    "best of panama",
]

all_scrub_terms = []
for v in COUNTRY_TERMS.values():    all_scrub_terms.extend(v)
for v in REGION_ALIASES.values():   all_scrub_terms.extend(v)
all_scrub_terms.extend(REGION_ONLY_TERMS)
all_scrub_terms.extend(CULTIVAR_TERMS)
all_scrub_terms.extend(PRODUCER_CONTEXT_TERMS)
all_scrub_terms = sorted(set(all_scrub_terms), key=len, reverse=True)
print(f'Loaded {len(all_scrub_terms)} scrub terms')

SCRUB_PATTERN = re.compile(
    r'\b(' + '|'.join(re.escape(t) for t in all_scrub_terms) + r')\b',
    flags=re.IGNORECASE,
)

def scrub(text):
    if not isinstance(text, str) or not text:
        return ''
    return SCRUB_PATTERN.sub(' [ORIGIN] ', text)


Loaded 403 scrub terms


## Build text + region; same split as NB 11


In [3]:
def minimal_raw_text(text):
    text = '' if pd.isna(text) else str(text)
    return re.sub(r'\s+', ' ', text).strip().lower()

def normalize_text_keep_brackets(text):
    text = minimal_raw_text(text)
    text = re.sub(r'[^a-z0-9\s\[\]]', ' ', text)
    return re.sub(r'\s+', ' ', text).strip()

EXTRA_TEXT_COLS = ['Blind Assessment', 'Notes', 'Who Should Drink It', 'Bottom Line']

def concat_and_scrub(row):
    pieces = []
    for col in EXTRA_TEXT_COLS:
        v = row[col]
        if pd.notna(v) and str(v).strip():
            raw = minimal_raw_text(v)
            scrubbed = scrub(raw)
            cleaned = normalize_text_keep_brackets(scrubbed)
            pieces.append(cleaned)
    return ' '.join(p for p in pieces if p)

df = pd.read_csv('Data/final_coffee_reviews.csv')
df['text_full_concat_scrubbed_plus'] = df.apply(concat_and_scrub, axis=1)
df['text_raw_minimal'] = df['Blind Assessment'].fillna('').map(minimal_raw_text)
df['origin_country'] = df['Country'].astype('string').str.strip().replace('', pd.NA)
df['origin_region']  = df['origin_country'].map(COUNTRY_TO_REGION)

work = df[
    (df['text_raw_minimal'].str.len() >= 30)
    & (df['origin_country'].notna())
    & (df['origin_region'].notna())
].copy().reset_index(drop=True)

y = work['origin_region']
X = work[TEXT_COLUMN].tolist()

X_tmp, X_test, y_tmp, y_test = train_test_split(X, y, test_size=0.15, stratify=y, random_state=RANDOM_STATE)
X_train, X_val, y_train, y_val = train_test_split(X_tmp, y_tmp, test_size=0.15/0.85, stratify=y_tmp, random_state=RANDOM_STATE)

label_names = sorted(y.unique().tolist())
label2id = {n: i for i, n in enumerate(label_names)}
id2label = {i: n for i, n in enumerate(label_names)}

train_texts, val_texts, test_texts = X_train, X_val, X_test
train_labels = [label2id[v] for v in y_train]
val_labels   = [label2id[v] for v in y_val]
test_labels  = [label2id[v] for v in y_test]

class_weights = compute_class_weight(class_weight='balanced',
                                     classes=np.arange(len(label_names)), y=train_labels)
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32)

print(f'Train/Val/Test: {len(train_labels)} / {len(val_labels)} / {len(test_labels)}')
print(f'Labels: {label_names}')
print(f'Class weights: {dict(zip(label_names, class_weights.round(3)))}')


Train/Val/Test: 5309 / 1138 / 1138
Labels: ['Asia-Pacific', 'Central America', 'East Africa', 'South America']
Class weights: {'Asia-Pacific': 1.784, 'Central America': 0.98, 'East Africa': 0.604, 'South America': 1.313}


## Dataset + run_one (RoBERTa-large)


In [4]:
class CoffeeOriginDataset(torch.utils.data.Dataset):
    def __init__(self, texts, labels, tokenizer, max_length):
        self.encodings = tokenizer(texts, truncation=True, max_length=max_length, padding=False)
        self.labels = labels
    def __len__(self): return len(self.labels)
    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='macro', zero_division=0)
    return {
        'accuracy': accuracy_score(labels, preds),
        'balanced_accuracy': balanced_accuracy_score(labels, preds),
        'precision_macro': precision, 'recall_macro': recall, 'f1_macro': f1,
    }

class WeightedTrainer(Trainer):
    def __init__(self, class_weights=None, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.pop('labels')
        outputs = model(**inputs)
        logits = outputs.get('logits')
        loss_fct = torch.nn.CrossEntropyLoss(weight=self.class_weights.to(logits.device))
        loss = loss_fct(logits, labels)
        return (loss, outputs) if return_outputs else loss

def run_one(seed, tag):
    set_seed(seed)
    out_dir = os.path.join(OUTPUT_DIR_ROOT, tag)

    tokenizer = AutoTokenizer.from_pretrained(ROBERTA_LARGE_CKPT)
    data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
    train_ds = CoffeeOriginDataset(train_texts, train_labels, tokenizer, MAX_LENGTH)
    val_ds   = CoffeeOriginDataset(val_texts,   val_labels,   tokenizer, MAX_LENGTH)
    test_ds  = CoffeeOriginDataset(test_texts,  test_labels,  tokenizer, MAX_LENGTH)

    model = AutoModelForSequenceClassification.from_pretrained(
        ROBERTA_LARGE_CKPT, num_labels=len(label_names),
        id2label=id2label, label2id=label2id,
    )

    args = TrainingArguments(
        output_dir=out_dir, learning_rate=ROBERTA_LARGE_LR,
        per_device_train_batch_size=TRAIN_BATCH_SIZE,
        per_device_eval_batch_size=EVAL_BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM_STEPS,
        num_train_epochs=12, weight_decay=WEIGHT_DECAY, warmup_ratio=WARMUP_RATIO,
        eval_strategy='epoch', logging_strategy='epoch',
        save_strategy='epoch', save_total_limit=1,
        load_best_model_at_end=True,
        metric_for_best_model='f1_macro', greater_is_better=True,
        disable_tqdm=True, report_to='none',
        fp16=USE_FP16 and torch.cuda.is_available(), seed=seed,
    )
    trainer = WeightedTrainer(
        model=model, args=args, class_weights=class_weights_tensor,
        train_dataset=train_ds, eval_dataset=val_ds,
        processing_class=tokenizer, data_collator=data_collator,
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
    )
    try: trainer.remove_callback(NotebookProgressCallback)
    except Exception: pass

    print(f'\n=== {tag} | roberta-large | lr={ROBERTA_LARGE_LR} | seed={seed} ===')
    trainer.train()
    val_m  = trainer.evaluate(eval_dataset=val_ds)
    test_m = trainer.evaluate(eval_dataset=test_ds, metric_key_prefix='test')

    del model, trainer
    if torch.cuda.is_available(): torch.cuda.empty_cache()

    return {
        'tag': tag, 'seed': seed,
        'val_f1_macro':  val_m['eval_f1_macro'],
        'val_bal_acc':   val_m['eval_balanced_accuracy'],
        'val_accuracy':  val_m['eval_accuracy'],
        'test_f1_macro': test_m['test_f1_macro'],
        'test_bal_acc':  test_m['test_balanced_accuracy'],
        'test_accuracy': test_m['test_accuracy'],
    }


## RoBERTa-large × 3 seeds


In [5]:
large_results = []
for s in SEEDS:
    large_results.append(run_one(seed=s, tag=f'roberta_large_region4way_seed{s}'))

large_df = pd.DataFrame(large_results)[['seed','val_f1_macro','test_f1_macro','test_bal_acc','test_accuracy']]
print()
print('RoBERTa-large per-seed:')
print(large_df.round(4).to_string(index=False))
print()
print('Mean ± Std:')
print(large_df.drop(columns=['seed']).agg(['mean','std']).round(4).to_string())

print()
print('--- Head-to-head ---')
print(f"RoBERTa-base   (NB 11): F1 0.8171 ± 0.0071  acc 0.8345 ± 0.0048")
print(f'RoBERTa-large  (NB 16): F1 {large_df["test_f1_macro"].mean():.4f} ± {large_df["test_f1_macro"].std():.4f}'
      f'  acc {large_df["test_accuracy"].mean():.4f} ± {large_df["test_accuracy"].std():.4f}')
print(f'Δ F1: {large_df["test_f1_macro"].mean() - 0.8171:+.4f}')
print(f'Δ acc: {large_df["test_accuracy"].mean() - 0.8345:+.4f}')


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
classifier.out_proj.weight      | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.dense.weight         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.



=== roberta_large_region4way_seed42 | roberta-large | lr=1e-05 | seed=42 ===
{'loss': '5.666', 'grad_norm': '61.81', 'learning_rate': '7.85e-06', 'epoch': '1'}
{'eval_loss': '1.357', 'eval_accuracy': '0.3805', 'eval_balanced_accuracy': '0.302', 'eval_precision_macro': '0.3141', 'eval_recall_macro': '0.302', 'eval_f1_macro': '0.2358', 'eval_runtime': '4.871', 'eval_samples_per_second': '233.6', 'eval_steps_per_second': '14.78', 'epoch': '1'}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'loss': '4.431', 'grad_norm': '198.6', 'learning_rate': '9.319e-06', 'epoch': '2'}
{'eval_loss': '0.7779', 'eval_accuracy': '0.732', 'eval_balanced_accuracy': '0.6826', 'eval_precision_macro': '0.7054', 'eval_recall_macro': '0.6826', 'eval_f1_macro': '0.6909', 'eval_runtime': '4.66', 'eval_samples_per_second': '244.2', 'eval_steps_per_second': '15.45', 'epoch': '2'}


...

{'train_runtime': '1101', 'train_samples_per_second': '57.87', 'train_steps_per_second': '1.809', 'train_loss': '2.158', 'epoch': '9'}


There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

{'eval_loss': '0.5968', 'eval_accuracy': '0.826', 'eval_balanced_accuracy': '0.8079', 'eval_precision_macro': '0.8116', 'eval_recall_macro': '0.8079', 'eval_f1_macro': '0.8095', 'eval_runtime': '5.029', 'eval_samples_per_second': '226.3', 'eval_steps_per_second': '14.32', 'epoch': '9'}


early stopping required metric_for_best_model, but did not find eval_f1_macro so early stopping is disabled


{'test_loss': '0.505', 'test_accuracy': '0.8366', 'test_balanced_accuracy': '0.8215', 'test_precision_macro': '0.8217', 'test_recall_macro': '0.8215', 'test_f1_macro': '0.8204', 'test_runtime': '4.712', 'test_samples_per_second': '241.5', 'test_steps_per_second': '15.28', 'epoch': '9'}

RoBERTa-large per-seed:
 seed  val_f1_macro  test_f1_macro  test_bal_acc  test_accuracy
   42        0.8242         0.8346        0.8367         0.8515
  123        0.8055         0.8425        0.8468         0.8550
 2024        0.8095         0.8204        0.8215         0.8366

Mean ± Std:
      val_f1_macro  test_f1_macro  test_bal_acc  test_accuracy
mean        0.8131         0.8325        0.8350         0.8477
std         0.0098         0.0112        0.0128         0.0098

--- Head-to-head ---
RoBERTa-base   (NB 11): F1 0.8171 ± 0.0071  acc 0.8345 ± 0.0048
RoBERTa-large  (NB 16): F1 0.8325 ± 0.0112  acc 0.8477 ± 0.0098
Δ F1: +0.0154
Δ acc: +0.0132


## Save results


In [6]:
out = {
    'notebook': '16_Origin_Regional_4Way_RoBERTaLarge',
    'task': '4-way regional, RoBERTa-large 3 seeds',
    'model': ROBERTA_LARGE_CKPT,
    'config': {
        'lr': ROBERTA_LARGE_LR,
        'train_batch_size': TRAIN_BATCH_SIZE,
        'grad_accum_steps': GRAD_ACCUM_STEPS,
        'effective_batch_size': TRAIN_BATCH_SIZE * GRAD_ACCUM_STEPS,
        'warmup_ratio': WARMUP_RATIO,
        'weighted_ce': True,
        'epochs': 12,
        'early_stop_patience': 3,
    },
    'seeds': SEEDS,
    'runs': large_results,
    'mean': large_df.drop(columns=['seed']).mean().to_dict(),
    'std':  large_df.drop(columns=['seed']).std().to_dict(),
    'reference_roberta_base_nb11': {
        'test_f1_macro_mean': 0.8171, 'test_f1_macro_std': 0.0071,
        'test_accuracy_mean': 0.8345, 'test_accuracy_std': 0.0048,
    },
}
out_path = os.path.join(OUTPUT_DIR_ROOT, 'results.json')
with open(out_path, 'w') as f:
    json.dump(out, f, indent=2, default=float)
print('Saved:', out_path)


Saved: artifacts/origin_region_4way_roberta_large\results.json
